# Comparative Architecture Analysis & Hyperparameter Optimization

Comprehensive evaluation comparing YOLOv8n and YOLO26n performance, featuring advanced fine-tuning via RayTune to identify the optimal model structure and parameter set.

1. Dataset correction — RGB → Grayscale conversion
2. Experiment — Model architecture comparison (YOLOv8s vs YOLO26s)
3. Hyperparameter tuning (Ray Tune / Ultralytics)
4. Final evaluation — Best tuned model on test set

In [1]:
import itertools, csv, time
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
import yaml

from pathlib import Path
from ultralytics import YOLO
from PIL import Image

### The "RGB" Revelation: 

The thermal images are black and white, but they were saved as 3-channel RGB files (copying the same gray data into Red, Green, and Blue).

In [ ]:
DATASET_ROOT = "./F_Binary_Dataset"
splits = ["train", "valid", "test"] # Adjust if your folders are named 'val' instead of 'valid'

print("🔄 Starting conversion to true grayscale...")

total_converted = 0

for split in splits:
    img_dir = Path(DATASET_ROOT) / split / "images"
    
    if not img_dir.exists():
        print(f"⚠️ Skipped '{split}': Directory not found at {img_dir}")
        continue
        
    files = list(img_dir.glob("*.jpg")) + list(img_dir.glob("*.png")) + list(img_dir.glob("*.jpeg"))
    
    if not files:
        print(f"⚠️ Skipped '{split}': No images found.")
        continue

    print(f"   Processing {split} ({len(files)} images)...")
    
    for img_path in files:
        try:
            img = Image.open(img_path)
            
            # Only convert if it's actually RGB
            if img.mode == 'RGB':
                gray_img = img.convert('L') # Convert to 8-bit grayscale (1 channel)
                gray_img.save(img_path)     # Overwrite original!
                total_converted += 1
        except Exception as e:
            print(f"❌ Error converting {img_path}: {e}")

print(f"\n✅ Conversion Complete!")
print(f"   Converted {total_converted} images to true 1-channel grayscale.")
print(f"   Dataset is now optimized for thermal segmentation.")

🔄 Starting conversion to true grayscale...
   Processing train (411 images)...
   Processing valid (127 images)...
   Processing test (95 images)...

✅ Conversion Complete!
   Converted 633 images to true 1-channel grayscale.
   Dataset is now optimized for thermal segmentation.


In [2]:
# repartition of the images in the dataset
dataset_path = "./F_Binary_Dataset"

for split in ["train", "valid", "test"]:
    img_dir = os.path.join(dataset_path, split, "images")
    print(split, "images:", len(os.listdir(img_dir)))

train images: 411
valid images: 127
test images: 95


In [3]:

# Path to one of your images
img_path = "./F_Binary_Dataset/train/images/video_001_event1_A002_500_t-00000s_01_jpg.rf.8a958210d0c7c29fd721f77f48ca9876.jpg" 

img = Image.open(img_path)
width, height = img.size
channels = len(img.getbands()) # 1 = Grayscale, 3 = RGB

print(f"✅ Image Size: {width} x {height} pixels")
print(f"✅ Color Channels: {channels} ({'Grayscale' if channels == 1 else 'RGB'})")

✅ Image Size: 432 x 432 pixels
✅ Color Channels: 1 (Grayscale)


### Yolov8 vs Yolo26

In [4]:
best_baseline = YOLO("./runs/segment/baseline_yolov8n/weights/best.pt")

In [6]:
model_26 = YOLO("yolo26n-seg.pt")

In [7]:
# train the model on our dataset
results = model_26.train(
    data="./F_Binary_Dataset/data.yaml",
    epochs=50,
    imgsz=640,
    batch=8,
    workers=4,
    name="yolo26n-seg"
)

New https://pypi.org/project/ultralytics/8.4.60 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.19 🚀 Python-3.10.19 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4090, 24215MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./F_Binary_Dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, n

In [5]:
# Paths to your trained models
best_26 = "./runs/segment/yolo26n-seg/weights/best.pt" 

In [ ]:
# ── 1. Configuration ──────────────────────────────────────────────────────────      

# Both models can be evaluated on the same grayscale test set for fair comparison.
DATASET_YAML = "./F_Binary_Dataset/data.yaml" 

print(f"🚀 Loading models and evaluating on Test Set...")

# ── 2. Define Evaluation Function ─────────────────────────────────────────────
def evaluate_model(name, model_path):
    try:
        model = YOLO(model_path)
        
        # Run validation on the TEST set (unseen data)
        # plots=False speeds up the process since we just want numbers
        metrics = model.val(data=DATASET_YAML, split="test", conf=0.25, iou=0.3, verbose=False, plots=False)
        
        return {
            "Model": name,
            "mAP50-95 (Box)": f"{metrics.box.map:.4f}",
            "mAP50-95 (Mask)": f"{metrics.seg.map:.4f}", # Primary metric for segmentation
            "Recall (Mask)": f"{metrics.seg.r[0]:.4f}",   # How many whales found?
            "Precision (Mask)": f"{metrics.seg.p[0]:.4f}",# How many detections were correct?
            "False Negatives (Est)": int(297 * (1 - metrics.seg.r[0])) # Assuming 297 test instances
        }
    except Exception as e:
        print(f"❌ Error evaluating {name}: {e}")
        return None

# ── 3. Run Evaluation ─────────────────────────────────────────────────────────
results = []

# Evaluate 1 model
print(f"   📊 Evaluating yolo26n-seg Model")
res_rgb = evaluate_model("yolo26n-seg", best_26)
if res_rgb: results.append(res_rgb)

# Evaluate the other model
print(f"   📊 Evaluating yolov8n-seg model")
res_gray = evaluate_model("yolov8n-seg", best_baseline)
if res_gray: results.append(res_gray)

# ── 4. Display Comparison Table ───────────────────────────────────────────────
if len(results) == 2:
    df_compare = pd.DataFrame(results)
    
    print("\n" + "="*80)
    print("🏆 HEAD-TO-HEAD COMPARISON (Test Set)")
    print("="*80)
    print(df_compare.to_string(index=False))
    print("="*80)
    
    # Simple logic to declare a winner
    mask_map_26s = float(df_compare[df_compare['Model'].str.contains('26n-seg')]['mAP50-95 (Mask)'].values[0])
    mask_map_v8s = float(df_compare[df_compare['Model'].str.contains('v8n-seg')]['mAP50-95 (Mask)'].values[0])
    
    if mask_map_26s > mask_map_v8s: 
        print(f"✅ WINNER: yolo26n-seg Model! (Improvement: {mask_map_26s - mask_map_v8s:.4f} mAP)")
    else:
        print(f"⚠️ RESULT: yolov8n-seg Model performed similarly or better. (Diff: {mask_map_26s - mask_map_v8s:.4f})")
        
else:
    print("❌ Could not complete comparison. Check file paths.")

🚀 Loading models and evaluating on Test Set...
   📊 Evaluating yolo26n-seg Model
Ultralytics 8.4.19 🚀 Python-3.10.19 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4090, 24215MiB)


YOLO26n-seg summary (fused): 139 layers, 2,689,079 parameters, 0 gradients, 9.0 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 642.4±198.5 MB/s, size: 10.2 KB)
val: Scanning /home/floreM/FlukePrint_YOLO/Gihub_final/F_Binary_Dataset/test/labels... 95 images, 10 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 95/95 1.7Kit/s 0.1s
val: New cache created: /home/floreM/FlukePrint_YOLO/Gihub_final/F_Binary_Dataset/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 7.4it/s 0.8s0.3s
                   all         95        297      0.756      0.596      0.694      0.549      0.778      0.613      0.714      0.514
Speed: 1.0ms preprocess, 2.8ms inference, 0.0ms loss, 1.0ms postprocess per image
   📊 Evaluating yolov8n-seg model
Ultralytics 8.4.19 🚀 Python-3.10.19 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4090, 24215MiB)
YOLOv8n-seg summary (fused): 86 layers, 3,258,2

### Hyperparameter Search with Ray Tune

In [6]:

model = YOLO("yolov8s-seg.pt")

model.tune(
    data="./F_Binary_Dataset/data.yaml",
    epochs=30,           
    iterations=50,       
    optimizer="AdamW",
    plots=False,
    save=False,
    val=True,
)

Tuner: Initialized Tuner instance with 'tune_dir=/home/floreM/FlukePrint_YOLO/Gihub_final/runs/segment/tune'
Tuner: 💡 Learn about tuning at https://docs.ultralytics.com/guides/hyperparameter-tuning
Tuner: Starting iteration 1/50 with hyperparameters: {'lr0': 0.01, 'lrf': 0.01, 'momentum': 0.937, 'weight_decay': 0.0005, 'warmup_epochs': 3.0, 'warmup_momentum': 0.8, 'box': 7.5, 'cls': 0.5, 'dfl': 1.5, 'hsv_h': 0.015, 'hsv_s': 0.7, 'hsv_v': 0.4, 'degrees': 0.0, 'translate': 0.1, 'scale': 0.5, 'shear': 0.0, 'perspective': 0.0, 'flipud': 0.0, 'fliplr': 0.5, 'bgr': 0.0, 'mosaic': 1.0, 'mixup': 0.0, 'cutmix': 0.0, 'copy_paste': 0.0, 'close_mosaic': 10}
New https://pypi.org/project/ultralytics/8.4.60 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.19 🚀 Python-3.10.19 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4090, 24215MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None

In [7]:
# Charger les meilleurs hyperparamètres
with open("./runs/segment/tune/best_hyperparameters.yaml") as f:
    best_params = yaml.safe_load(f)

# Entraînement final avec ces paramètres
model = YOLO("yolov8s-seg.pt")

model.train(
    data="./F_Binary_Dataset/data.yaml",
    epochs=100,      
    imgsz=432,
    **best_params    
)

# Évaluation finale — une seule fois — sur le test set
model.val(data="./F_Binary_Dataset/data.yaml", split="test", conf=0.25, iou=0.3)

New https://pypi.org/project/ultralytics/8.4.60 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.19 🚀 Python-3.10.19 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4090, 24215MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.00235, box=7.40493, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.76144, compile=False, conf=None, copy_paste=0.00441, copy_paste_mode=flip, cos_lr=False, cutmix=0.00452, data=./F_Binary_Dataset/data.yaml, degrees=0.0076, deterministic=True, device=None, dfl=2.84763, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.39206, flipud=0.0003, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.00441, hsv_s=0.74073, hsv_v=0.20363, imgsz=432, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.00064, lrf=0.01175, mask_ratio=4, max_det=300, mixup=0.00077, mode=train, model=yolov8

ultralytics.utils.metrics.SegmentMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7f74c65a1450>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)', 'Precision-Recall(M)', 'F1-Confidence(M)', 'Precision-Confidence(M)', 'Recall-Confidence(M)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041, 

In [8]:
# ── 1. Configuration ──────────────────────────────────────────────────────────
MODEL_BASELINE_PATH = "./runs/segment/baseline_yolov8n/weights/best.pt"
MODEL_TUNED_PATH    = "./runs/segment/tune/weights/best.pt"

DATASET_YAML   = "./F_Binary_Dataset/data.yaml"
N_TEST_IMAGES  = 95

print("🚀 Loading models and evaluating on Test Set...")

# ── 2. Define Evaluation Function ─────────────────────────────────────────────
def evaluate_model(name, model_path):
    try:
        model = YOLO(model_path)
        metrics = model.val(
            data=DATASET_YAML, split="test",
            conf=0.25, iou=0.3,
            verbose=False, plots=False
        )
        return {
            "Model":                  name,
            "mAP50-95 (Box)":         f"{metrics.box.map:.4f}",
            "mAP50-95 (Mask)":        f"{metrics.seg.map:.4f}",
            "Recall (Mask)":          f"{metrics.seg.r.mean():.4f}",
            "Precision (Mask)":       f"{metrics.seg.p.mean():.4f}",
            "False Negatives (Est)":  int(N_TEST_IMAGES * (1 - metrics.seg.r.mean()))
        }
    except Exception as e:
        print(f"❌ Error evaluating {name}: {e}")
        return None

# ── 3. Run Evaluation ─────────────────────────────────────────────────────────
results = []

print("   📊 Evaluating Baseline (yolov8s default params)...")
res_baseline = evaluate_model("Baseline (default)", MODEL_BASELINE_PATH)
if res_baseline: results.append(res_baseline)

print("   📊 Evaluating Tuned (best hyperparameters)...")
res_tuned = evaluate_model("Tuned (best params)", MODEL_TUNED_PATH)
if res_tuned: results.append(res_tuned)

# ── 4. Display Comparison Table ───────────────────────────────────────────────
if len(results) == 2:
    df_compare = pd.DataFrame(results)

    print("\n" + "="*80)
    print("🏆 BASELINE vs TUNED — Test Set Comparison")
    print("="*80)
    print(df_compare.to_string(index=False))
    print("="*80)

    map_baseline = float(df_compare[df_compare["Model"].str.contains("Baseline")]["mAP50-95 (Mask)"].values[0])
    map_tuned    = float(df_compare[df_compare["Model"].str.contains("Tuned")]   ["mAP50-95 (Mask)"].values[0])
    diff         = map_tuned - map_baseline

    if diff > 0:
        print(f"✅ WINNER: Tuned model! (mAP improvement: +{diff:.4f})")
    elif diff < 0:
        print(f"⚠️  RESULT: Baseline performed better. (mAP diff: {diff:.4f})")
    else:
        print(f"🤝 RESULT: Both models are equivalent. (mAP diff: {diff:.4f})")

else:
    print("❌ Could not complete comparison. Check file paths.")

🚀 Loading models and evaluating on Test Set...
   📊 Evaluating Baseline (yolov8s default params)...
Ultralytics 8.4.19 🚀 Python-3.10.19 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4090, 24215MiB)


YOLOv8n-seg summary (fused): 86 layers, 3,258,259 parameters, 0 gradients, 11.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 857.0±247.2 MB/s, size: 11.1 KB)
val: Scanning /home/floreM/FlukePrint_YOLO/Gihub_final/F_Binary_Dataset/test/labels.cache... 95 images, 10 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 95/95 26.6Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 12.9it/s 0.5s.1s
                   all         95        297      0.837      0.779      0.839      0.656      0.837      0.779      0.842      0.588
Speed: 0.9ms preprocess, 1.5ms inference, 0.0ms loss, 0.6ms postprocess per image
   📊 Evaluating Tuned (best hyperparameters)...
Ultralytics 8.4.19 🚀 Python-3.10.19 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4090, 24215MiB)
YOLOv8s-seg summary (fused): 86 layers, 11,779,987 parameters, 0 gradients, 39.9 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.